# Transformer (Attention-Based) - PyTorch

In [3]:
"""
Transformer Text Classification on SST-2
----------------------------------------

Features:
- PyTorch 2.x compatible
- Python 3.12 compatible
- No torchtext
- Transformer Encoder
- Proper attention masking
- Positional Encoding
- SST-2 Dataset
"""

import math
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from datasets import load_dataset
from sklearn.metrics import accuracy_score


# ============================================================
# 1. CONFIG
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VOCAB_SIZE = 25000
MAX_LEN = 64

EMBED_DIM = 128
NUM_HEADS = 4
FF_DIM = 256
NUM_LAYERS = 2
DROPOUT = 0.1

BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 1e-4

PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"

print("Using device:", DEVICE)


# ============================================================
# 2. LOAD DATASET
# ============================================================

dataset = load_dataset("glue", "sst2")

train_dataset = dataset["train"]
validation_dataset = dataset["validation"]

print("Train samples:", len(train_dataset))
print("Validation samples:", len(validation_dataset))


# ============================================================
# 3. TOKENIZER
# ============================================================

def tokenizer(text):
    return text.lower().split()


# ============================================================
# 4. BUILD VOCAB
# ============================================================

counter = Counter()

for sample in train_dataset:
    counter.update(tokenizer(sample["sentence"]))

most_common = counter.most_common(VOCAB_SIZE - 2)
vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}

for idx, (word, _) in enumerate(most_common, start=2):
    vocab[word] = idx

PAD_IDX = vocab[PAD_TOKEN]
UNK_IDX = vocab[UNK_TOKEN]

print("Vocabulary size:", len(vocab))


# ============================================================
# 5. ENCODING
# ============================================================

def encode_text(text):
    tokens = tokenizer(text)
    token_ids = [vocab.get(token, UNK_IDX) for token in tokens]
    token_ids = token_ids[:MAX_LEN]

    return torch.tensor(token_ids, dtype=torch.long)


# ============================================================
# 6. COLLATE FUNCTION
# ============================================================

def collate_batch(batch):
    texts = []
    labels = []

    for sample in batch:
        text_tensor = encode_text(sample["sentence"])
        label = sample["label"]
        pad_length = MAX_LEN - len(text_tensor)

        if pad_length > 0:
            padding = torch.full((pad_length,), PAD_IDX, dtype=torch.long)
            text_tensor = torch.cat([text_tensor, padding])

        texts.append(text_tensor)
        labels.append(label)

    texts = torch.stack(texts)
    labels = torch.tensor(labels, dtype=torch.float32)

    return texts.to(DEVICE), labels.to(DEVICE)


# ============================================================
# 7. DATALOADERS
# ============================================================

train_loader = DataLoader(train_dataset,
                          batch_size=BATCH_SIZE,
                          shuffle=True,
                          collate_fn=collate_batch)

validation_loader = DataLoader(validation_dataset,
                               batch_size=BATCH_SIZE,
                               shuffle=False,
                               collate_fn=collate_batch)


# ============================================================
# 8. POSITIONAL ENCODING
# ============================================================

class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)

        self.register_buffer("pe", pe)

    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len]


# ============================================================
# 9. TRANSFORMER MODEL
# ============================================================

class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, ff_dim, num_layers, max_len, dropout=0.1):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.positional_encoding = PositionalEncoding(embed_dim, max_len)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True
        )

        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        padding_mask = (x == PAD_IDX)

        x = self.embedding(x)
        x = x * math.sqrt(EMBED_DIM)
        x = self.positional_encoding(x)

        x = self.transformer_encoder(x, src_key_padding_mask=padding_mask)

        # Masked Mean Pooling
        mask = (~padding_mask).unsqueeze(-1)
        x = x * mask
        summed = x.sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1)
        x = summed / counts

        x = self.dropout(x)
        logits = self.classifier(x)

        return logits.squeeze(1)


# ============================================================
# 10. INIT MODEL
# ============================================================

model = TransformerClassifier(
    vocab_size=len(vocab),
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    ff_dim=FF_DIM,
    num_layers=NUM_LAYERS,
    max_len=MAX_LEN,
    dropout=DROPOUT
).to(DEVICE)

print(model)


# ============================================================
# 11. LOSS + OPTIMIZER
# ============================================================

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)


# ============================================================
# 12. TRAINING
# ============================================================

def train_epoch(model, dataloader):
    model.train()
    total_loss = 0

    for texts, labels in dataloader:
        optimizer.zero_grad()

        outputs = model(texts)
        loss = criterion(outputs, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


# ============================================================
# 13. EVALUATION
# ============================================================

def evaluate(model, dataloader):
    model.eval()

    predictions = []
    true_labels = []
    total_loss = 0

    with torch.no_grad():
        for texts, labels in dataloader:
            outputs = model(texts)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            probs = torch.sigmoid(outputs)
            preds = (probs >= 0.5).int()

            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(true_labels, predictions)
    avg_loss = total_loss / len(dataloader)

    return avg_loss, accuracy


# ============================================================
# 14. TRAIN LOOP
# ============================================================

print("\nStarting training...\n")

for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, validation_loader)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss : {train_loss:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Val Acc    : {val_acc:.4f}")
    print("-" * 50)


# ============================================================
# 15. INFERENCE
# ============================================================

def predict_sentiment(model, text):
    model.eval()

    with torch.no_grad():
        tokens = encode_text(text)

        if len(tokens) < MAX_LEN:
            padding = torch.full(
                (MAX_LEN - len(tokens),),
                PAD_IDX,
                dtype=torch.long
            )
            tokens = torch.cat([tokens, padding])

        tokens = tokens.unsqueeze(0).to(DEVICE)
        output = model(tokens)
        prob = torch.sigmoid(output).item()

        sentiment = "Positive" if prob >= 0.5 else "Negative"

    return sentiment, prob


# ============================================================
# 16. TEST EXAMPLES
# ============================================================

sample_texts = [
    "This movie was quietly brilliant.",
    "The plot was not bad but not memorable either.",
    "Absolutely terrible acting.",
    "I expected more from this film."
]

print("\nSample Predictions:\n")

for text in sample_texts:
    sentiment, confidence = predict_sentiment(model, text)

    print(f"Text       : {text}")
    print(f"Prediction : {sentiment}")
    print(f"Confidence : {confidence:.4f}")
    print("-" * 50)

Using device: cuda
Train samples: 67349
Validation samples: 872
Vocabulary size: 14818
TransformerClassifier(
  (embedding): Embedding(14818, 128, padding_idx=0)
  (positional_encoding): PositionalEncoding()
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (dropout): Dropout(p=0.1, inplace=False)
  (classifier): Sequential(
    

```
How to adapt this implementation:

1. Replace IMDB dataset with:
       - AG_NEWS
       - Yelp Reviews
       - Custom CSV/Text dataset
       - HuggingFace datasets

2. Multi-class classification:
       - Change final layer:
             nn.Linear(64, num_classes)

       - Use:
             CrossEntropyLoss()

3. Larger Transformer:
       - Increase NUM_LAYERS
       - Increase NUM_HEADS
       - Increase EMBED_DIM

4. Better pooling:
       - CLS token pooling
       - Attention pooling
       - Max pooling

5. Add learning rate scheduler:
       torch.optim.lr_scheduler

6. Use pretrained embeddings:
       - GloVe
       - FastText
       - BERT embeddings
```